In [1]:
import polars as pl
import pyprojroot

ROOT = pyprojroot.here()

Saqué un par de variables de acá https://www.kaggle.com/code/thiagomantuani/predict-hits-with-new-mlb-get-started-here/notebook

In [ ]:
FT_TO_CM = 30.48
HALF_PLATE_CM = 21.59   # mitad de las 17 pulgadas del plato → 8.5 in * 2.54 cm/in
SHADOW_MARGIN_CM = 5.08  # 2 pulgadas fuera del borde de zona



def aplicar_feature_engineering(df: pl.DataFrame, is_train: bool = True) -> pl.DataFrame:
    """
    Toma un DataFrame crudo y lo devuelve aplicando transformaciones.
    """
    
    # Conversión de unidades (pies a cm)
    df = df.with_columns( 
        plate_x = pl.col("plate_x") * FT_TO_CM,
        plate_z = pl.col("plate_z") * FT_TO_CM,
        pfx_x   = pl.col("pfx_x")   * FT_TO_CM,
        pfx_z   = pl.col("pfx_z")   * FT_TO_CM,
        sz_top  = pl.col("sz_top")  * FT_TO_CM, 
        sz_bot  = pl.col("sz_bot")  * FT_TO_CM
    )

    # Variable respuesta (solo disponible para el conjunto de entrenamiento)
    swing_values = [
        "swinging_strike", "swinging_strike_blocked",
        "foul", "foul_tip", "foul_bunt", "bunt_foul_tip",
        "missed_bunt", "hit_into_play", "foul_pitchout"
        ]

    if is_train and "description" in df.columns:
        df = df.with_columns(
            swing = pl.col("description").is_in(swing_values).cast(pl.Int8)  
        ).drop('description')

    # Primeras métricas base de zona y movimiento 
    df = df.with_columns( 
        sz_mid = (pl.col("sz_top") + pl.col("sz_bot")) / 2, # Centro vertical de la zona
        strike_zone_size = pl.col("sz_top") - pl.col("sz_bot"), # Altura total de la zona
        movement_complexity = (pl.col("pfx_x")**2 + pl.col("pfx_z")**2).sqrt(), # Magnitud total
        platoon_advantage = (pl.col("p_throws") != pl.col("stand")).cast(pl.Int8) # Ventaja
    )

    # Localización normalizada
    df = df.with_columns(
        relative_height = ( # Vertical: 0 = piso de zona, 1 = techo
            (pl.col("plate_z") - pl.col("sz_bot")) / pl.col("strike_zone_size")
        ),
        relative_x = pl.col("plate_x") / HALF_PLATE_CM # Horizontal
    )

    # Indicadoras de zona de strike
    df = df.with_columns(
        is_strike_zone = (
            pl.col("plate_x").is_between(-HALF_PLATE_CM, HALF_PLATE_CM) &
            pl.col("relative_height").is_between(0, 1)
        ).cast(pl.Int8)
    )
    
    df = df.with_columns(
        is_shadow_zone = (
            pl.col("plate_x").is_between(
                -HALF_PLATE_CM - SHADOW_MARGIN_CM,
                 HALF_PLATE_CM + SHADOW_MARGIN_CM
                 ) &
            pl.col("plate_z").is_between(
                pl.col("sz_bot") - SHADOW_MARGIN_CM,
                pl.col("sz_top") + SHADOW_MARGIN_CM
            ) &
            (pl.col("is_strike_zone") == 0)
        ).cast(pl.Int8)
        )

    # Distancia al centro de zona
    df = df.with_columns(
        pitch_location = (
            pl.col("plate_x")**2 +
            (pl.col("plate_z") - pl.col("sz_mid"))**2
        ).sqrt(),
    )

    # Distancia al rincón más cercano (mide qué tan esquinado fue)
    df = df.with_columns(
        distance_to_corner = pl.when(pl.col("is_strike_zone") == 1).then(
            (
                (1 - pl.col("relative_x").abs())**2 +
                pl.min_horizontal(
                    pl.col("relative_height"),        # distancia al piso (0)
                    1 - pl.col("relative_height"),    # distancia al techo (1)
                )**2
            ).sqrt()
        ).otherwise(pl.lit(0.0))
    )

    # Correción de bolas y strikes
    df = df.with_columns(
        balls = pl.when(pl.col('balls') == 4).then(pl.lit(3)).otherwise('balls'),
        strikes = pl.when(pl.col('strikes') == 3).then(pl.lit(2)).otherwise('strikes')
    )

    # Interacciones
    df = df.with_columns( 
        complex_speed = pl.col("release_speed") * pl.col("movement_complexity"),
        count = pl.col("balls").cast(pl.Utf8) + pl.lit("-") + pl.col("strikes").cast(pl.Utf8)
    )


    # Variables categóricas
    cat_cols = [
        'batter', 'pitcher', 'stand', 'p_throws', 'pitch_type', 'count'
        ]
        
    df = df.with_columns([
        pl.col(c).cast(pl.String).cast(pl.Categorical) for c in cat_cols if c in df.columns
    ])

    return df

In [ ]:
FT_TO_CM = 30.48
HALF_PLATE_CM = 21.59   # mitad de las 17 pulgadas del plato → 8.5 in * 2.54 cm/in
SHADOW_MARGIN_CM = 5.08  # 2 pulgadas fuera del borde de zona

def aplicar_feature_engineering_2(df, is_train):
    """
    Toma un DataFrame crudo y lo devuelve aplicando transformaciones.
    """
    
    # Conversión de unidades (pies a cm)
    df = df.with_columns( 
        plate_x = pl.col("plate_x") * FT_TO_CM,
        plate_z = pl.col("plate_z") * FT_TO_CM,
        pfx_x   = pl.col("pfx_x")   * FT_TO_CM,
        pfx_z   = pl.col("pfx_z")   * FT_TO_CM,
        sz_top  = pl.col("sz_top")  * FT_TO_CM, 
        sz_bot  = pl.col("sz_bot")  * FT_TO_CM
    )

    # Variable respuesta (solo disponible para el conjunto de entrenamiento)
    swing_values = [
        "swinging_strike", "swinging_strike_blocked",
        "foul", "foul_tip", "foul_bunt", "bunt_foul_tip",
        "missed_bunt", "hit_into_play", "foul_pitchout"
        ]

    if is_train and "description" in df.columns:
        df = df.with_columns(
            swing = pl.col("description").is_in(swing_values).cast(pl.Int8)  
        ).drop('description')

     
    # sz_bot, sz_top y todas las variables derivadas de estas 2 "rompían" los modelos
    # Como la zona de strike varía de bateador en bateador, y no es constante para un mismo bateador, la hacemos constante calculando su promedio.
    zona_promedio = df.group_by("batter").agg(
        sz_top_mean = pl.col("sz_top").mean(),
        sz_bot_mean = pl.col("sz_bot").mean()
        )
    
    # Cruzamos los promedios al dataset
    df = df.join(zona_promedio, on="batter", how="left").drop('sz_bot', 'sz_top')
    
    df = df.with_columns(
        strike_zone_size = pl.col("sz_top_mean") - pl.col("sz_bot_mean"),
        sz_mid = (pl.col("sz_top_mean") + pl.col("sz_bot_mean"))/2,
        relative_x = pl.col("plate_x") / HALF_PLATE_CM
    )

    df = df.with_columns(
        # 2. relative_height depende de strike_zone_size calculado arriba
        relative_height = (pl.col("plate_z") - pl.col("sz_bot_mean")) / pl.col("strike_zone_size")
    )

    df = df.with_columns(
        # 3. Indicadora de si pasó por la zona de strike 'promedio' de ese bateador
        is_strike_zone = (
            (pl.col("plate_x") >= -HALF_PLATE_CM) & 
            (pl.col("plate_x") <= HALF_PLATE_CM) & 
            (pl.col("relative_height") >= 0) & 
            (pl.col("relative_height") <= 1)
        ).cast(pl.Int8) # Casteamos a entero igual que el .astype(int)
    )

    # Primeras métricas base de zona y movimiento 
    df = df.with_columns( 
        movement_complexity = (pl.col("pfx_x")**2 + pl.col("pfx_z")**2).sqrt(), # Magnitud total
        platoon_advantage = (pl.col("p_throws") != pl.col("stand")).cast(pl.Int8) # Ventaja
    )

    df = df.with_columns(
        # Tomamos en cuenta la circunferencia de la pelota
        is_shadow_zone = (
            pl.col("plate_x").is_between(-HALF_PLATE_CM - SHADOW_MARGIN_CM, HALF_PLATE_CM + SHADOW_MARGIN_CM) &
            pl.col("plate_z").is_between(
                pl.col("sz_bot_mean") - SHADOW_MARGIN_CM,
                pl.col("sz_top_mean") + SHADOW_MARGIN_CM
            ) &
            (pl.col("is_strike_zone") == 0)
        ).cast(pl.Int8)
    )

    # Distancia al centro de zona
    df = df.with_columns(
        pitch_location = (
            pl.col("plate_x")**2 +
            (pl.col("plate_z") - pl.col("sz_mid"))**2
        ).sqrt(),
    )

    # Distancia al rincón más cercano (mide qué tan esquinado fue)
    df = df.with_columns(
        distance_to_corner = pl.when(pl.col("is_strike_zone") == 1).then(
            (
                (1 - pl.col("relative_x").abs())**2 +
                pl.min_horizontal(
                    pl.col("relative_height"),        # distancia al piso (0)
                    1 - pl.col("relative_height"),    # distancia al techo (1)
                )**2
            ).sqrt()
        ).otherwise(pl.lit(0.0))
    )

    # Correción de bolas y strikes
    df = df.with_columns(
        balls = pl.when(pl.col('balls') == 4).then(pl.lit(3)).otherwise('balls'),
        strikes = pl.when(pl.col('strikes') == 3).then(pl.lit(2)).otherwise('strikes')
    )

    # Interacciones
    df = df.with_columns( 
        complex_speed = pl.col("release_speed") * pl.col("movement_complexity"),
        count = pl.col("balls").cast(pl.Utf8) + pl.lit("-") + pl.col("strikes").cast(pl.Utf8)
    )
    
    # Variables categóricas
    cat_cols = [
        'batter', 'pitcher', 'stand', 'p_throws', 'pitch_type', 'count'
        ]
        
    df = df.with_columns([
        pl.col(c).cast(pl.String).cast(pl.Categorical) for c in cat_cols if c in df.columns
    ])
   
    
    return df

### Procesamiento de los datasets originales

In [ ]:
t1_raw = pl.read_parquet(ROOT / "data" / "raw" / "temporada1.parquet")
t2_raw = pl.read_parquet(ROOT / "data" / "raw" / "temporada2.parquet")

t1_procesada = aplicar_feature_engineering(t1_raw, is_train=True)
t2_procesada = aplicar_feature_engineering(t2_raw, is_train=False)

### Guardado de archivos

In [5]:
path_procesados = ROOT / 'data' / 'processed'

# Temporada 1 CON nulos
t1_procesada.write_parquet(path_procesados / 'temporada1_con_nulos.parquet')

# Temporada 1 SIN nulos
t1_procesada.drop_nulls().write_parquet(path_procesados / 'temporada1_limpio.parquet')

# Temporada 2 CON nulos
t2_procesada.write_parquet(path_procesados / 'temporada2_limpio.parquet')